# Verify setup (read-only checks + live write-read-delete tests)
- Checks this company's 3 environments and the shared account items: storage, S3, catalogs, volumes, grants, users/groups/service principals, lock-down, Git folders and tokens.
- Run as admin: Run all. The last cell lists every WARN/FAIL. Safe to re-run. Not part of any job. Prints no emails, IDs or keys.

## 1. Constants

In [0]:
REPO_COMPANY = "fqf"
COMPANY_NAME = "Flexi Quant Finance"
REPO_NAME    = "flexi_quant_finance_fd"

ENVS         = ["dev", "test", "prod"]
S3_FOLDERS   = ["landing", "processed", "rejected"]
SCHEMA_NAMES = ["bronze", "silver", "gold", "ops"]
GROUPS       = ["lh_developer", "lh_teamlead", "lh_manager", "lh_bizmanager", "lh_management"]
MUST_HAVE_MEMBERS = ["lh_developer", "lh_teamlead"]          # others are added on 7 and 14 Oct
VOLUME_RULES = {
    "landing":     ("bronze", "EXTERNAL", {"dev", "test", "prod"}),
    "processed":   ("bronze", "EXTERNAL", {"dev", "test", "prod"}),
    "rejected":    ("bronze", "EXTERNAL", {"dev", "test", "prod"}),
    "checkpoints": ("ops",    "MANAGED",  {"dev", "test", "prod"}),
    "refresh_in":  ("ops",    "MANAGED",  {"dev", "test"}),
    "refresh_out": ("ops",    "MANAGED",  {"test", "prod"}),
}
R      = ["USE CATALOG", "USE SCHEMA", "SELECT", "READ VOLUME", "EXECUTE", "BROWSE"]
RW     = R + ["MODIFY", "CREATE TABLE", "CREATE VOLUME", "WRITE VOLUME", "CREATE FUNCTION", "CREATE MATERIALIZED VIEW"]
GOLD_R = ["USE SCHEMA", "SELECT", "EXECUTE"]
def matrix(env):
    sp = f"sp_{REPO_COMPANY}_{env}"
    return {
        "dev":  [("lh_developer", "catalog", RW), ("lh_teamlead", "catalog", RW),
                 ("lh_manager", "catalog", RW), ("lh_management", "catalog", R + ["MANAGE"])],
        "test": [("lh_developer", "catalog", R), ("lh_teamlead", "catalog", R),
                 ("lh_manager", "catalog", R + ["MANAGE"]), ("lh_bizmanager", "gold", GOLD_R),
                 ("lh_management", "catalog", R + ["MANAGE"]), (sp, "catalog", RW)],
        "prod": [("lh_developer", "catalog", R), ("lh_teamlead", "catalog", R),
                 ("lh_manager", "catalog", R), ("lh_bizmanager", "gold", GOLD_R),
                 ("lh_management", "catalog", R + ["MANAGE"]), (sp, "catalog", RW)],
    }[env]

from databricks.sdk import WorkspaceClient
import datetime
w = WorkspaceClient()
results = []
def check(area, env, name, ok, detail="", level="FAIL"):
    results.append((area, env, name, "PASS" if ok else level, str(detail)[:180]))
print(f"{COMPANY_NAME}: checking {', '.join(f'{REPO_COMPANY}_{e}' for e in ENVS)} + shared account items")

## 2. Storage credentials, external locations, S3 (live)

In [0]:
for env in ENVS:
    bucket = f"lh-{REPO_COMPANY}-{env}-source"
    cred   = f"lh_{REPO_COMPANY}_{env}_cred"
    loc    = f"lh_{REPO_COMPANY}_{env}_source"
    role   = f"lh-{REPO_COMPANY}-{env}-databricks"
    try:
        c = w.storage_credentials.get(cred)
        arn = c.aws_iam_role.role_arn if c.aws_iam_role else ""
        check("storage", env, f"credential {cred} exists", True)
        check("storage", env, "credential uses its own IAM role", arn.endswith(f":role/{role}"), "role " + arn.split(":role/")[-1])
        check("storage", env, "credential is not read-only", not c.read_only)
    except Exception as e:
        check("storage", env, f"credential {cred} exists", False, type(e).__name__)
    try:
        l = w.external_locations.get(loc)
        check("storage", env, f"external location {loc} exists", True)
        check("storage", env, "location points to own bucket", (l.url or "").rstrip("/") == f"s3://{bucket}", l.url)
        check("storage", env, "location uses own credential", l.credential_name == cred, l.credential_name)
        check("storage", env, "location is not read-only", not l.read_only)
    except Exception as e:
        check("storage", env, f"external location {loc} exists", False, type(e).__name__)
    try:
        top = {f.name.rstrip("/") for f in dbutils.fs.ls(f"s3://{bucket}/")}
        check("s3", env, "bucket reachable (live)", True)
        for f in S3_FOLDERS:
            check("s3", env, f"folder {f}/ exists", f in top)
        extra = sorted(top - set(S3_FOLDERS))
        check("s3", env, "no extra top-level folders/files", not extra, extra or "none")
    except Exception as e:
        check("s3", env, "bucket reachable (live)", False, type(e).__name__)
print("Storage and S3 checked")

## 3. Catalogs, schemas, volumes (+ live write-read-delete on every volume)

In [0]:
all_catalogs = {c.name for c in w.catalogs.list()}
for env in ENVS:
    cat = f"{REPO_COMPANY}_{env}"
    bucket = f"lh-{REPO_COMPANY}-{env}-source"
    if cat not in all_catalogs:
        check("catalog", env, f"catalog {cat} exists", False)
        continue
    check("catalog", env, f"catalog {cat} exists", True)
    schemas = {s.name for s in w.schemas.list(catalog_name=cat)} - {"information_schema"}
    check("catalog", env, "schemas exactly bronze/silver/gold/ops", schemas == set(SCHEMA_NAMES),
          f"missing {sorted(set(SCHEMA_NAMES) - schemas) or 'none'}, extra {sorted(schemas - set(SCHEMA_NAMES)) or 'none'}")
    check("catalog", env, "no 'default' schema", "default" not in schemas)
    rows = spark.sql(f"""SELECT volume_schema, volume_name, volume_type, storage_location
                         FROM system.information_schema.volumes WHERE volume_catalog = '{cat}'""").collect()
    actual   = {(r.volume_schema, r.volume_name): r for r in rows}
    expected = {(s, v) for v, (s, t, envs) in VOLUME_RULES.items() if env in envs}
    check("volume", env, "volumes exactly as designed", set(actual) == expected,
          f"missing {sorted(expected - set(actual)) or 'none'}, extra {sorted(set(actual) - expected) or 'none'}")
    for (s, v), r in actual.items():
        if v not in VOLUME_RULES:
            continue
        vtype = VOLUME_RULES[v][1]
        check("volume", env, f"{s}.{v} is {vtype}", r.volume_type == vtype, r.volume_type)
        if vtype == "EXTERNAL":
            check("volume", env, f"{s}.{v} on s3://{bucket}/{v}", (r.storage_location or "").rstrip("/") == f"s3://{bucket}/{v}")
        path = f"/Volumes/{cat}/{s}/{v}/_verify/verify.txt"
        try:
            dbutils.fs.put(path, "ok", True)
            ok = dbutils.fs.head(path) == "ok"
            dbutils.fs.rm(f"/Volumes/{cat}/{s}/{v}/_verify", True)
            check("volume", env, f"{s}.{v} write-read-delete (live)", ok)
        except Exception as e:
            check("volume", env, f"{s}.{v} write-read-delete (live)", False, type(e).__name__)
print("Catalogs, schemas and volumes checked")

## 4. Grants vs access matrix (catalog, every schema, every volume)

In [0]:
sp_app = {}
for env in ["test", "prod"]:
    name = f"sp_{REPO_COMPANY}_{env}"
    sps = list(w.service_principals.list(filter=f"displayName eq '{name}'"))
    if sps:
        sp_app[sps[0].application_id] = name
norm = lambda a: a.replace("_", " ").upper()
def grants(obj_type, obj):
    rows = spark.sql(f"SHOW GRANTS ON {obj_type} {obj}").collect()
    return {(sp_app.get(r.Principal, r.Principal), norm(r.ActionType)) for r in rows if r.ObjectType.upper() == obj_type}

for env in ENVS:
    cat = f"{REPO_COMPANY}_{env}"
    if cat not in all_catalogs:
        continue
    exp_cat, exp_gold = set(), set()
    for name, scope, privs in matrix(env):
        if scope == "catalog":
            exp_cat |= {(name, norm(p)) for p in privs}
        else:
            exp_cat |= {(name, "USE CATALOG"), (name, "BROWSE")}
            exp_gold |= {(name, norm(p)) for p in privs}
    act_cat = grants("CATALOG", cat)
    check("grants", env, "catalog grants match matrix", act_cat == exp_cat,
          f"missing {len(exp_cat - act_cat)}, extra {sorted(act_cat - exp_cat) or 'none'}")
    for s in SCHEMA_NAMES:
        act = grants("SCHEMA", f"{cat}.{s}")
        exp = exp_gold if s == "gold" else set()
        check("grants", env, f"schema {s} grants as designed", act == exp,
              f"missing {len(exp - act)}, extra {sorted(act - exp) or 'none'}")
    vol_rows = spark.sql(f"SELECT volume_schema, volume_name FROM system.information_schema.volumes WHERE volume_catalog = '{cat}'").collect()
    vol_extra = [f"{r.volume_schema}.{r.volume_name}" for r in vol_rows if grants("VOLUME", f"{cat}.{r.volume_schema}.{r.volume_name}")]
    check("grants", env, "no direct grants on volumes", not vol_extra, vol_extra or "none")
    writers = {p for p, a in act_cat if a in ("MODIFY", "CREATE TABLE", "WRITE VOLUME")}
    exp_writers = {n for n, sc, pr in matrix(env) if "MODIFY" in pr}
    check("grants", env, "only designed principals can write", writers == exp_writers, sorted(writers))
print("Grants checked")

## 5. Users, groups, service principals (counts only, no names)

In [0]:
def group(name):
    g = list(w.groups.list(filter=f"displayName eq '{name}'", attributes="id,displayName,members"))
    return g[0] if g else None

admins = group("admins")
admin_ids = {m.value for m in (admins.members or [])} if admins else set()
check("identity", "all", "workspace admins = 1 (you)", len(admin_ids) == 1, f"{len(admin_ids)} admin(s)", level="WARN")
for gname in GROUPS:
    g = group(gname)
    check("identity", "all", f"group {gname} exists", g is not None)
    if not g:
        continue
    ids = {m.value for m in (g.members or [])}
    check("identity", "all", f"{gname} has members", len(ids) >= 1, f"{len(ids)} member(s)",
          level="FAIL" if gname in MUST_HAVE_MEMBERS else "WARN")
    check("identity", "all", f"{gname} members are not admins", not (ids & admin_ids))
for env in ["test", "prod"]:
    name = f"sp_{REPO_COMPANY}_{env}"
    sps = list(w.service_principals.list(filter=f"displayName eq '{name}'"))
    check("identity", env, f"service principal {name} exists", bool(sps))
    if sps:
        sp = w.service_principals.get(sps[0].id)
        ent = sorted(e.value for e in (sp.entitlements or []))
        check("identity", env, f"{name} active", bool(sp.active))
        check("identity", env, f"{name} not admin", sp.id not in admin_ids)
        check("identity", env, f"{name} entitlements: workspace + SQL only",
              set(ent) <= {"workspace-access", "databricks-sql-access"} and "databricks-sql-access" in ent, ent)
print("Users, groups and service principals checked")

## 6. Lock-down (old catalogs and workspace catalog hidden; project catalogs not open to all)

In [0]:
bad = lambda p: p == "account users" or p.startswith("_workspace_users")
for cat in sorted(all_catalogs):
    if cat.startswith(("bhg_", "fqf_")) or cat in ("system", "samples"):
        continue
    try:
        open_to = sorted({r.Principal for r in spark.sql(f"SHOW GRANTS ON CATALOG `{cat}`").collect() if bad(r.Principal)})
        check("lockdown", "all", f"catalog {cat} hidden from all users", not open_to, open_to or "none")
    except Exception as e:
        check("lockdown", "all", f"catalog {cat} hidden from all users", False, type(e).__name__, level="WARN")
for env in ENVS:
    cat = f"{REPO_COMPANY}_{env}"
    if cat in all_catalogs:
        open_to = sorted({r.Principal for r in spark.sql(f"SHOW GRANTS ON CATALOG {cat}").collect() if bad(r.Principal)})
        check("lockdown", env, "catalog not open to all users", not open_to, open_to or "none")
print("Lock-down checked")

## 7. Git folders, tokens, GitHub link

In [0]:
from databricks.sdk.service.workspace import ObjectType
me   = w.current_user.me().user_name
home = f"/Users/{me}"

def branch_of(path):
    """Branch if this edition exposes it, else None."""
    try:
        st = w.workspace.get_status(path)
        return w.repos.get(repo_id=st.object_id).branch
    except Exception:
        return None

for env in ENVS:
    branch = f"{REPO_COMPANY}_{env}"
    fname  = REPO_NAME if env == "dev" else f"{REPO_NAME}_{env}"
    path   = f"{home}/{fname}"
    try:
        exists = w.workspace.get_status(path).object_type in (ObjectType.DIRECTORY, ObjectType.REPO)
    except Exception:
        exists = False
    check("git", env, f"Git folder {fname} exists", exists)
    if not exists:
        continue
    try:
        nbs = {o.path.split("/")[-1] for o in w.workspace.list(f"{path}/setup_once/notebook")}
    except Exception:
        nbs = set()
    need = {"00_setup_environment", "01_grant_access"}
    check("git", env, f"{fname} has both setup notebooks", need <= nbs, sorted(nbs) or "none")
    b = branch_of(path)
    if b is None:
        check("git", env, f"{fname} is on {branch}", False, "branch not exposed by API on this edition: confirm in UI", level="WARN")
    else:
        check("git", env, f"{fname} is on {branch}", b == branch, b)

now_ms = datetime.datetime.now().timestamp() * 1000
toks = [t for t in w.tokens.list() if (t.comment or "") == f"lh_github_{REPO_COMPANY}"]
check("tokens", "all", f"token lh_github_{REPO_COMPANY} exists", len(toks) == 1, f"{len(toks)} found")
if toks:
    exp = toks[0].expiry_time or 0
    days = int((exp - now_ms) / 86400000) if exp > 0 else 9999
    check("tokens", "all", "token valid for 14+ days", days >= 14, f"{days} days left", level="WARN")
creds = list(w.git_credentials.list())
providers = [str(getattr(c, "git_provider", "")) for c in creds]
check("git", "all", "GitHub credential linked", any("github" in p.lower() for p in providers), providers)
print("Git folders and tokens checked")

In [0]:
df = spark.createDataFrame(results, "area string, env string, check string, status string, detail string")
display(df.orderBy("status", "area", "env"))
total = len(results)
counts = {s: sum(1 for r in results if r[3] == s) for s in ("PASS", "WARN", "FAIL")}
print(f"{COMPANY_NAME}: {total} checks | PASS {counts['PASS']} | WARN {counts['WARN']} | FAIL {counts['FAIL']}")
for r in results:
    if r[3] != "PASS":
        print(f"  {r[3]:4} [{r[0]}/{r[1]}] {r[2]} -> {r[4]}")